In [4]:
from getData import tms_to_geotiff
from PIL import Image
import numpy as np
import argparse
import cv2
import ee

from sam2.build_sam import build_sam2
import torch
from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator
import matplotlib.pyplot as plt
from scipy.ndimage import binary_erosion, label

from osgeo import gdal, osr
from ultralytics import YOLO

# Initialize Earth Engine
ee.Initialize()

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_0JLhFqfSY1uiEaW?source=Init


In [2]:
# ft = ee.FeatureCollection("projects/myanmar-crops/assets/fieldBoundaries/sailinGridv4sub")

# # Define argument parser
# parser = argparse.ArgumentParser(description="Process a specific feature by index.")
# parser.add_argument("--nr", type=int, required=True, help="Index of the feature to process")

# # Parse arguments
# args = parser.parse_args()
# # nr = args.nr
# nr = 1

# name = str(ee.Feature(ft.toList(5000).get(nr)).get("grid_id").getInfo())

# # Extract coordinates
# coordinates = ee.Feature(ft.toList(5000).get(nr)).geometry().getInfo()['coordinates'][0]

# # Calculate xmin, xmax, ymin, ymax
# xmin = min(coord[0] for coord in coordinates)
# xmax = max(coord[0] for coord in coordinates)
# ymin = min(coord[1] for coord in coordinates)
# ymax = max(coord[1] for coord in coordinates)

# bbox = [xmin, ymin, xmax, ymax]
# print("Name:", name)
# print("Bounding Box:", bbox)

In [ ]:
# test mocking command with notbook
import sys
import argparse
ft = ee.FeatureCollection("projects/myanmar-crops/assets/fieldBoundaries/sailinGridv4sub")

# Mock command-line arguments
sys.argv = ['script_name', '--nr', '']  # Replace '0' with your desired index

# Define argument parser
parser = argparse.ArgumentParser(description="Process a specific feature by index.")
parser.add_argument("--nr", type=int, required=True, help="Index of the feature to process")

# Parse arguments
args = parser.parse_args()
nr = args.nr

name = str(ee.Feature(ft.toList(5000).get(nr)).get("grid_id").getInfo())

# Extract coordinates
coordinates = ee.Feature(ft.toList(5000).get(nr)).geometry().getInfo()['coordinates'][0]

# Calculate xmin, xmax, ymin, ymax
xmin = min(coord[0] for coord in coordinates)
xmax = max(coord[0] for coord in coordinates)
ymin = min(coord[1] for coord in coordinates)
ymax = max(coord[1] for coord in coordinates)

bbox = [xmin, ymin, xmax, ymax]
print("Name:", name)
print("Bounding Box:", bbox)

Name: N205121E946058
Bounding Box: [94.60578220443497, 20.512089001233267, 94.62578216922901, 20.53208908293838]


In [7]:
# outfolder = "D:/SIG/Sam2_soucre/Test_pipeline/"
# image = f"{outfolder}{name}.tif"
# satImg = tms_to_geotiff(output=image, bbox=bbox, zoom=19, source="Satellite", overwrite=True, return_image=True)
# print('image name:',image)
# print('satImg:',satImg)

In [1]:
# ## Check GPU
# # import torch
# print("CUDA Available:", torch.cuda.is_available())
# print("CUDA Version:", torch.version.cuda)
# print("Device Name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU detected")

In [3]:
# Load YOLO model
yolo_model = YOLO("D:/SIG/Yolo/training_results/field_detection_exp11/weights/best.pt")

# Run YOLO prediction
# yolo_results = yolo_model.predict(source=f"{outfolder}{name}.tif", save=False)
yolo_results = yolo_model.predict(source='D:/SIG/Sam2_soucre/Test_pipeline/img/N205121E946058.tif', save=False)

# Extract bounding boxes
bounding_boxes = []
for result in yolo_results:
    for box in result.boxes.xyxy:
        bounding_boxes.append(box.cpu().numpy())
        print(box)


image 1/1 D:\SIG\Sam2_soucre\Test_pipeline\img\N205121E946058.tif: 640x608 2 fields, 199.9ms
Speed: 13.6ms preprocess, 199.9ms inference, 13.0ms postprocess per image at shape (1, 3, 640, 608)
tensor([   3.9863,    0.0000, 1798.1738,  714.4285])
tensor([2.9039e+03, 3.4701e-01, 4.0193e+03, 5.8121e+02])


In [4]:
def convert_yolo_boxes_to_sam2(bounding_boxes, img_width, img_height):
    sam2_boxes = []
    for box in bounding_boxes:
        x_min, y_min, x_max, y_max = box
        sam2_boxes.append({
            "x_min": int(x_min),
            "y_min": int(y_min),
            "x_max": int(x_max),
            "y_max": int(y_max)
        })
    return sam2_boxes


In [6]:
def simplify_contours(binary_mask, epsilon_factor=0.005):
    """
    Simplify contours in a binary mask.
    Args:
    - binary_mask: Binary mask (numpy array) with 0s and positive values.
    - epsilon_factor: Proportion of the contour perimeter to use as the approximation accuracy.

    Returns:
    - Simplified binary mask.
    """
    # Ensure the mask is binary and of type uint8
    binary_mask = (binary_mask > 0).astype(np.uint8) * 255  # Convert to 0 and 255
    contours, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    simplified_mask = np.zeros_like(binary_mask)

    for contour in contours:
        epsilon = epsilon_factor * cv2.arcLength(contour, True)
        simplified_contour = cv2.approxPolyDP(contour, epsilon, True)
        cv2.drawContours(simplified_mask, [simplified_contour], -1, 255, thickness=cv2.FILLED)

    return (simplified_mask > 0).astype(np.uint16)


In [ ]:
# import cv2
# import numpy as np
# import matplotlib.pyplot as plt

# result_image = cv2.imread("D:/SIG/Yolo/predict/field_detection_m/640csailin428_areas_shp_1.jpg", cv2.IMREAD_GRAYSCALE)

# _, binary_mask = cv2.threshold(result_image, 127, 255, cv2.THRESH_BINARY)


# simplified_mask = simplify_contours(binary_mask, epsilon_factor=0.01)

# plt.figure(figsize=(10, 5))
# plt.subplot(1, 2, 1)
# plt.title("Original Mask from Prediction")
# plt.imshow(binary_mask, cmap="gray")
# plt.subplot(1, 2, 2)
# plt.title("Simplified Mask")
# plt.imshow(simplified_mask, cmap="gray")
# plt.show()

In [ ]:
def split_image_with_overlap(image, tile_size, overlap):
    height, width, _ = image.shape
    tiles = []
    step = tile_size - overlap

    for y in range(0, height, step):
        for x in range(0, width, step):
            y_end = min(y + tile_size, height)
            x_end = min(x + tile_size, width)
            tile = image[y:y_end, x:x_end]
            tiles.append((tile, x, y, x_end, y_end))

    return tiles

# Split the image
tiles = split_image_with_overlap(data, tile_size=640, overlap=140)


In [ ]:
for idx, (tile, x, y, x_end, y_end) in enumerate(tiles):
    predictor.set_image(tile)
    sam2_boxes = convert_yolo_boxes_to_sam2(bounding_boxes, tile.shape[1], tile.shape[0])

    all_masks = []
    for box in sam2_boxes:
        box_coords = [box['x_min'], box['y_min'], box['x_max'], box['y_max']]
        mask, score, low_res_mask = predictor.predict(box=box_coords)
        all_masks.append({
            "mask": mask,
            "score": score,
            "low_res_mask": low_res_mask,
            "box": box_coords
        })


In [ ]:
def save_as_geotiff(output_path, mask_array, bbox, width, height):
    lon_min, lat_min, lon_max, lat_max = bbox
    geotransform = [
        lon_min, (lon_max - lon_min) / width, 0,
        lat_max, 0, -(lat_max - lat_min) / height
    ]

    driver = gdal.GetDriverByName('GTiff')
    dataset = driver.Create(output_path, width, height, 1, gdal.GDT_UInt16)
    dataset.SetGeoTransform(geotransform)

    srs = osr.SpatialReference()
    srs.ImportFromEPSG(4326)
    dataset.SetProjection(srs.ExportToWkt())

    dataset.GetRasterBand(1).WriteArray(mask_array)
    dataset.GetRasterBand(1).SetNoDataValue(0)
    dataset.FlushCache()
    dataset = None

# Save the final combined mask
save_as_geotiff(f"{outfolder}{name}_fields.tif", combined_mask, bbox, data.shape[1], data.shape[0])
